# Experiments and Model Train

This notebook runs all training conditions and generates the figures for the paper.

**Conditions:** Baseline | $SO(2)$ with $m \in \{1,2,4,8,16\}$ | $C_4$ with $m \in \{1,2,4,8,16\}$

In [ ]:
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from haar_experiments import run_all_experiments, DEVICE
from haar_plots import *

print(f"Device: {DEVICE}")
print(f"CWD: {os.getcwd()}")

In [ ]:
QUICK_MODE  = False 
LOAD_CACHED = False 

RESULTS_FILE = "results.pkl"

cfg = dict(
    m_values  = [1, 2, 4, 8, 16],
    ck_orders = [4, 8], 
    epochs   = 2 if QUICK_MODE else 15,
    n_train  = 2000 if QUICK_MODE else 8000,
    n_test   = 500 if QUICK_MODE else 2000,
    seed     = 42,
    save_path= RESULTS_FILE,
    resume   = True,
)

print("Configuration:")
for k, v in cfg.items():
    print(f"  {k:12s} = {v}")

Configuration:
  m_values     = [1, 2, 4, 8, 16]
  ck_orders    = [4, 8]
  epochs       = 15
  n_train      = 8000
  n_test       = 2000
  seed         = 42
  save_path    = results.pkl
  resume       = True


In [ ]:
if LOAD_CACHED and os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "rb") as f:
        results = pickle.load(f)
    print(f"Loaded {len(results)} cached conditions from {RESULTS_FILE!r}")
    for key, v in results.items():
        print(f"  {key:20s}  acc={v['final_acc']:.4f}  ece={v['ece']:.4f}  "
              f"rob={v['mean_robustness']:.4f}")
else:
    results = run_all_experiments(**cfg)
    print(f"Done. {len(results)} conditions saved to {RESULTS_FILE!r}")


────────────────────────────────────────────────────────────
  Condition : baseline
  Epochs    : 15   |   Train N : 8000   |   Device : cpu
────────────────────────────────────────────────────────────
  ep 01/15  loss=0.6822  acc_clean=0.9480  acc_rotated=0.3400  ovar=0.00000  train=4.9s
  ep 02/15  loss=0.2122  acc_clean=0.9645  acc_rotated=0.3645  ovar=0.00000  train=4.7s
  ep 03/15  loss=0.1487  acc_clean=0.9780  acc_rotated=0.3715  ovar=0.00000  train=5.1s
  ep 04/15  loss=0.1100  acc_clean=0.9770  acc_rotated=0.3875  ovar=0.00000  train=4.9s
  ep 05/15  loss=0.0896  acc_clean=0.9750  acc_rotated=0.3700  ovar=0.00000  train=4.8s
  ep 06/15  loss=0.0740  acc_clean=0.9795  acc_rotated=0.3885  ovar=0.00000  train=4.7s
  ep 07/15  loss=0.0565  acc_clean=0.9795  acc_rotated=0.3865  ovar=0.00000  train=4.7s
  ep 08/15  loss=0.0510  acc_clean=0.9810  acc_rotated=0.3810  ovar=0.00000  train=4.7s
  ep 09/15  loss=0.0456  acc_clean=0.9780  acc_rotated=0.3705  ovar=0.00000  train=4.9s
  ep 

In [ ]:
print(f"{'Condition':<20}  {'Acc (rot)':<10}  {'Rob':<8}  {'ECE':<8}  {'Train (s)':<10}")
print("-" * 64)

order = (["baseline"]
         + [f"so2_m{m}" for m in [1,2,4,8,16]]
         + [f"c4_m{m}"  for m in [1,2,4,8,16]]
         + [f"c8_m{m}"  for m in [1,2,4,8,16]])

for key in order:
    if key not in results:
        continue
    r = results[key]
    t = r.get("total_train_time_s", float("nan"))
    print(f"{key:<20} {r['final_acc']:.4f} {r['mean_robustness']:.4f} {r['ece']:.4f}  {t:.1f}s")

Condition             Acc (rot)   Rob       ECE       Train (s) 
----------------------------------------------------------------
baseline              0.4240      0.4004    0.3851  136.0s
so2_m1                0.8530      0.8477    0.0825  134.3s
so2_m2                0.8755      0.8744    0.0620  195.6s
so2_m4                0.8940      0.8900    0.0550  294.3s
so2_m8                0.8980      0.8983    0.0391  532.3s
so2_m16               0.8895      0.8916    0.0450  964.8s
c4_m1                 0.7510      0.7554    0.0334  140.5s
c4_m2                 0.7470      0.7516    0.0282  196.4s
c4_m4                 0.7695      0.7725    0.0297  300.7s
c4_m8                 0.7610      0.7717    0.0278  535.5s
c4_m16                0.7690      0.7737    0.0316  954.8s
c8_m1                 0.8405      0.8394    0.0810  139.4s
c8_m2                 0.8815      0.8786    0.0569  195.3s
c8_m4                 0.8835      0.8831    0.0582  299.1s
c8_m8                 0.8900      0.8941    